In [33]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv("../data/processed/clean_terai_heatwave.csv")
df.head()

,Date,Year,Day_of_Year,Province,District,Regional_Hub,Latitude,Longitude,Max_Temp_C,Min_Temp_C,Mean_Temp_C,Humidity_Pct,Wind_Speed_ms,Precipitation_mm
0,2026-09-17,2026,260,Bagmati,Chitwan,Bharatpur,27.68,84.43,29.43,24.65,26.43,91.49,1.15,12.56
1,2026-09-17,2026,260,Gandaki,Nawalpur,Kawasoti,27.65,84.12,29.43,24.65,26.43,91.49,1.15,12.56
2,2026-09-17,2026,260,Koshi,Jhapa,Bhadrapur,26.54,88.08,30.02,24.77,26.82,90.43,0.92,9.43
3,2026-09-17,2026,260,Koshi,Morang,Biratnagar,26.45,87.27,30.55,25.11,27.26,89.39,0.73,8.12
4,2026-09-17,2026,260,Koshi,Sunsari,Inaruwa,26.60,87.15,31.03,25.35,27.71,88.39,0.78,7.52


### Sort by District and Date

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 204897 entries, 0 to 204896
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Date              204897 non-null  str    
 1   Year              204897 non-null  int64  
 2   Day_of_Year       204897 non-null  int64  
 3   Province          204897 non-null  str    
 4   District          204897 non-null  str    
 5   Regional_Hub      204897 non-null  str    
 6   Latitude          204897 non-null  float64
 7   Longitude         204897 non-null  float64
 8   Max_Temp_C        204897 non-null  float64
 9   Min_Temp_C        204897 non-null  float64
 10  Mean_Temp_C       204897 non-null  float64
 11  Humidity_Pct      204897 non-null  float64
 12  Wind_Speed_ms     204897 non-null  float64
 13  Precipitation_mm  204897 non-null  float64
dtypes: float64(8), int64(2), str(4)
memory usage: 28.2 MB


In [5]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(by=["District", "Date"]).reset_index(drop=True)
df[["District", "Date", "Max_Temp_C", "Min_Temp_C", "Precipitation_mm"]].head()

,District,Date,Max_Temp_C,Min_Temp_C,Precipitation_mm
0,Banke,2000-01-01,24.02,10.40,0.0
1,Banke,2000-01-02,24.44,11.22,0.0
2,Banke,2000-01-03,23.54,11.55,0.0
3,Banke,2000-01-04,23.82,10.08,0.0
4,Banke,2000-01-05,23.82,10.57,0.0


### Heatwave Falg
- heatwave is more or less that 40


In [6]:
df["Heatwave_Flag"] = (df["Max_Temp_C"] >= 40.0).astype(int)
print(df["Heatwave_Flag"].value_counts())
print(f"\nHeatwave Percentage: {df['Heatwave_Flag'].mean() * 100:.2f}% of total days")

Heatwave_Flag
0    191693
1     13204
Name: count, dtype: int64

Heatwave Percentage: 6.44% of total days


### Heatwave Severity
- the heatlevel with normal,moderate,severe,and extreme

In [7]:
conditions = [
    df["Max_Temp_C"] < 40.0,                                    # 0: Normal
    (df["Max_Temp_C"] >= 40.0) & (df["Max_Temp_C"] < 42.0),     # 1: Moderate
    (df["Max_Temp_C"] >= 42.0) & (df["Max_Temp_C"] < 45.0),     # 2: Severe
    df["Max_Temp_C"] >= 45.0                                    # 3: Extreme
]
choices = [0, 1, 2, 3]

df["Heatwave_Severity"] = np.select(conditions, choices, default=0).astype(int)
severity_labels = {
    0: "0: Normal (<40°C)",
    1: "1: Moderate (40-42°C)",
    2: "2: Severe (42-45°C)",
    3: "3: Extreme (>=45°C)"   
}
print(df["Heatwave_Severity"].map(severity_labels).value_counts().sort_index())

Heatwave_Severity
0: Normal (<40°C)        191693
1: Moderate (40-42°C)      8205
2: Severe (42-45°C)        4600
3: Extreme (>=45°C)         399
Name: count, dtype: int64


### Tomorrow's High & Low
- next day temprature max and min

In [8]:

df["Next_Day_Max_Temp"] = df.groupby("District")["Max_Temp_C"].shift(-1)
df["Next_Day_Min_Temp"] = df.groupby("District")["Min_Temp_C"].shift(-1)

sample_cols = ["District", "Date", "Max_Temp_C", "Next_Day_Max_Temp", "Min_Temp_C", "Next_Day_Min_Temp"]
df[df["District"] == "Banke"][sample_cols].head(4)


,District,Date,Max_Temp_C,Next_Day_Max_Temp,Min_Temp_C,Next_Day_Min_Temp
0,Banke,2000-01-01,24.02,24.44,10.40,11.22
1,Banke,2000-01-02,24.44,23.54,11.22,11.55
2,Banke,2000-01-03,23.54,23.82,11.55,10.08
3,Banke,2000-01-04,23.82,23.82,10.08,10.57


### Temprature Range
- gap between max and min


In [9]:

df["Temp_Range"] = df["Max_Temp_C"] - df["Min_Temp_C"]

print(df["Temp_Range"].describe().round(2))
df[["District", "Date", "Max_Temp_C", "Min_Temp_C", "Temp_Range"]].head()


count    204897.00
mean         10.96
std           3.83
min           0.97
25%           7.63
50%          11.31
75%          13.94
max          23.18
Name: Temp_Range, dtype: float64


,District,Date,Max_Temp_C,Min_Temp_C,Temp_Range
0,Banke,2000-01-01,24.02,10.40,13.62
1,Banke,2000-01-02,24.44,11.22,13.22
2,Banke,2000-01-03,23.54,11.55,11.99
3,Banke,2000-01-04,23.82,10.08,13.74
4,Banke,2000-01-05,23.82,10.57,13.25


### Ambient Temperature vs. Apparent "Feels-Like" Temperature (Steadman Heat Index)
- actual tempratur ourbody feels 

In [10]:


# 1. Daytime Atmospheric Vapor Pressure (hPa) & Daytime Feels-Like
e_a_max = (df["Humidity_Pct"] / 100.0) * 6.1078 * np.exp((17.27 * df["Max_Temp_C"]) / (df["Max_Temp_C"] + 237.3))
df["Feels_Like_Max_C"] = df["Max_Temp_C"] + (0.33 * e_a_max) - (0.70 * df["Wind_Speed_ms"]) - 4.00

# 2. Nighttime Atmospheric Vapor Pressure (hPa) & Nighttime Feels-Like
e_a_min = (df["Humidity_Pct"] / 100.0) * 6.1078 * np.exp((17.27 * df["Min_Temp_C"]) / (df["Min_Temp_C"] + 237.3))
df["Feels_Like_Min_C"] = df["Min_Temp_C"] + (0.33 * e_a_min) - (0.70 * df["Wind_Speed_ms"]) - 4.00

# 3. Both Day and Night Thermal Discomfort Gaps
df["Day_Thermal_Gap"] = (df["Feels_Like_Max_C"] - df["Max_Temp_C"]).round(2)
df["Night_Thermal_Gap"] = (df["Feels_Like_Min_C"] - df["Min_Temp_C"]).round(2)

summary_cols = [
    "Max_Temp_C", "Feels_Like_Max_C", "Day_Thermal_Gap",
    "Min_Temp_C", "Feels_Like_Min_C", "Night_Thermal_Gap"
]
print(df[summary_cols].describe().round(2))
df[["District", "Date"] + summary_cols].head()


       Max_Temp_C  Feels_Like_Max_C  Day_Thermal_Gap  Min_Temp_C  \
count   204897.00         204897.00        204897.00   204897.00   
mean        30.88             34.15             3.26       19.92   
std          5.63              7.31             3.45        6.15   
min         11.77             10.08            -5.36        0.07   
25%         26.79             28.15             0.20       14.52   
50%         30.77             35.72             3.03       21.68   
75%         34.71             39.82             6.65       25.21   
max         48.80             57.62            11.88       34.71   

       Feels_Like_Min_C  Night_Thermal_Gap  
count         204897.00          204897.00  
mean              19.60              -0.32  
std                8.31               2.85  
min               -4.27              -6.52  
25%               12.15              -2.81  
50%               20.50              -1.03  
75%               27.55               2.46  
max               34.78    

,District,Date,Max_Temp_C,Feels_Like_Max_C,Day_Thermal_Gap,Min_Temp_C,Feels_Like_Min_C,Night_Thermal_Gap
0,Banke,2000-01-01,24.02,21.197228,-2.82,10.40,6.379278,-4.02
1,Banke,2000-01-02,24.44,21.611414,-2.83,11.22,7.171454,-4.05
2,Banke,2000-01-03,23.54,20.515745,-3.02,11.55,7.450217,-4.10
3,Banke,2000-01-04,23.82,20.812267,-3.01,10.08,5.876004,-4.20
4,Banke,2000-01-05,23.82,20.714744,-3.11,10.57,6.328640,-4.24


### Thermodynamic Wet-Bulb Temperature
- how effectively evaporation can cool the air/body.


In [11]:

T = df["Max_Temp_C"]
RH = df["Humidity_Pct"]
df["Wet_Bulb_Temp"] = (
    T * np.arctan(0.151977 * np.sqrt(RH + 8.313659))
    + np.arctan(T + RH)
    - np.arctan(RH - 1.676331)
    + 0.00391838 * (RH ** 1.5) * np.arctan(0.023101 * RH)
    - 4.686035
).round(2)

# Calculate the Wet-Bulb Evaporative Depression Gap
df["Wet_Bulb_Depression"] = (df["Max_Temp_C"] - df["Wet_Bulb_Temp"]).round(2)

print(df["Wet_Bulb_Temp"].describe().round(2))

danger_days = (df["Wet_Bulb_Temp"] >= 30.0).sum()
extreme_days = (df["Wet_Bulb_Temp"] >= 32.0).sum()
print(f"\nDays with Wet-Bulb >= 30°C (High Danger): {danger_days:,}")
print(f"Days with Wet-Bulb >= 32°C (Extreme Danger): {extreme_days:,}")

df[["District", "Date", "Max_Temp_C", "Humidity_Pct", "Wet_Bulb_Temp", "Wet_Bulb_Depression"]].sample(5)


count    204897.00
mean         23.83
std           5.14
min           8.38
25%          19.50
50%          24.58
75%          28.52
max          36.45
Name: Wet_Bulb_Temp, dtype: float64

Days with Wet-Bulb >= 30°C (High Danger): 17,515
Days with Wet-Bulb >= 32°C (Extreme Danger): 637


,District,Date,Max_Temp_C,Humidity_Pct,Wet_Bulb_Temp,Wet_Bulb_Depression
96906,Kapilvastu,2024-11-23,25.56,67.21,21.06,4.50
158770,Rupandehi,2007-04-12,36.43,33.40,24.10,12.33
42925,Dang,2010-09-02,25.66,90.41,24.40,1.26
159048,Rupandehi,2008-01-15,21.63,49.53,15.03,6.60
48503,Dang,2025-12-10,21.06,51.03,14.75,6.31


### Nocturnal Thermal Stress (Tropical Night Indicator >= 25°C)
- the temperature does not fall below 25°C during the night.


In [12]:
# Step 7: Atmospheric Physics - Tropical Night Indicator (WMO Standard >= 25.0°C)
df["Tropical_Night"] = (df["Min_Temp_C"] >= 25.0).astype(int)

# Check distribution and frequency
total_tropical = df["Tropical_Night"].sum()
tropical_pct = df["Tropical_Night"].mean() * 100

print(f"Total Tropical Nights (Min Temp >= 25°C): {total_tropical:,} days")
print(f"Percentage of Total Days: {tropical_pct:.2f}%")

# Prove predictive impact: Average Tomorrow's Max Temp after a Tropical Night vs Normal Night
avg_temp_comparison = df.groupby("Tropical_Night")["Next_Day_Max_Temp"].mean().round(2)


print(f"After Normal Night (Min < 25°C)  : {avg_temp_comparison.get(0, 'N/A')}°C")
print(f"After Tropical Night (Min >= 25°C): {avg_temp_comparison.get(1, 'N/A')}°C")

# Sample rows
df[["District", "Date", "Min_Temp_C", "Tropical_Night", "Max_Temp_C", "Next_Day_Max_Temp"]].sample(5)


Total Tropical Nights (Min Temp >= 25°C): 56,505 days
Percentage of Total Days: 27.58%
After Normal Night (Min < 25°C)  : 29.25°C
After Tropical Night (Min >= 25°C): 35.19°C


,District,Date,Min_Temp_C,Tropical_Night,Max_Temp_C,Next_Day_Max_Temp
168472,Saptari,2007-02-16,10.69,0,22.35,23.55
118065,Nawalpur,2002-09-08,25.36,1,36.87,36.72
71724,Kailali,2009-05-18,27.62,1,43.15,42.27
125205,Nawalpur,2022-03-27,19.64,0,35.42,35.24
62393,Jhapa,2010-07-18,25.07,1,29.26,30.36


### Scorching Westerly Wind Index
- How hot, dry, and windy the air conditions are.


In [13]:

df["Loo_Index"] = (
    (df["Max_Temp_C"] * df["Wind_Speed_ms"]) / (df["Humidity_Pct"] + 1.0)
).round(2)

print(df["Loo_Index"].describe().round(2))

# Prove regional divide: Check which districts suffer the highest Loo Index
# print('district')
top_loo_districts = df.groupby("District")["Loo_Index"].mean().sort_values(ascending=False).head(5)
print(top_loo_districts.round(2))

# Sample 3 rows
df[["District", "Date", "Max_Temp_C", "Wind_Speed_ms", "Humidity_Pct", "Loo_Index"]].sample(5)


count    204897.00
mean          1.25
std           1.40
min           0.10
25%           0.49
50%           0.77
75%           1.43
max          25.50
Name: Loo_Index, dtype: float64
District
Kapilvastu    1.54
Bardiya       1.45
Dhanusha      1.41
Mahottari     1.41
Banke         1.41
Name: Loo_Index, dtype: float64


,District,Date,Max_Temp_C,Wind_Speed_ms,Humidity_Pct,Loo_Index
38247,Chitwan,2024-07-29,32.68,1.15,82.20,0.45
108010,Morang,2001-11-14,28.63,0.88,53.92,0.46
116595,Morang,2025-05-17,33.50,2.58,64.73,1.31
110501,Morang,2008-09-09,29.91,1.33,88.67,0.44
47297,Dang,2022-08-22,30.63,2.74,78.84,1.05


### Atmospheric Memory: Autoregressive Lags (Yesterday's Weather & Lag-2)


In [14]:

df["Yesterday_Max_Temp"] = df.groupby("District")["Max_Temp_C"].shift(1)
df["Yesterday_Min_Temp"] = df.groupby("District")["Min_Temp_C"].shift(1)
df["Yesterday_Precip"]   = df.groupby("District")["Precipitation_mm"].shift(1)
df["Yesterday_Humidity"] = df.groupby("District")["Humidity_Pct"].shift(1)

check_cols = ["District", "Date", "Yesterday_Max_Temp", "Max_Temp_C","Yesterday_Min_Temp","Min_Temp_C","Yesterday_Humidity","Humidity_Pct", "Yesterday_Precip", "Precipitation_mm"]
df[df["District"] == "Banke"][check_cols].head(4)


,District,Date,Yesterday_Max_Temp,Max_Temp_C,Yesterday_Min_Temp,Min_Temp_C,Yesterday_Humidity,Humidity_Pct,Yesterday_Precip,Precipitation_mm
0,Banke,2000-01-01,NaN,24.02,NaN,10.40,NaN,21.03,NaN,0.0
1,Banke,2000-01-02,24.02,24.44,10.40,11.22,21.03,21.35,0.0,0.0
2,Banke,2000-01-03,24.44,23.54,11.22,11.55,21.35,21.15,0.0,0.0
3,Banke,2000-01-04,23.54,23.82,11.55,10.08,21.15,21.11,0.0,0.0


### Day & Night Temperature Momentum
- gap in temprature of maximun and minimum in previous 1st and 2nd day

In [15]:
# Step 10: Thermal Momentum & Acceleration (Day & Night Velocity Delta T)

# 1. Nocturnal Lag-2 (Night baseline 2 days ago, grouped by District)
df["Max_Temp_Lag2"]= df.groupby("District")["Max_Temp_C"].shift(2)
df["Min_Temp_Lag2"]= df.groupby("District")["Min_Temp_C"].shift(2)

df["Temp_Momentum_1D"] = df["Yesterday_Max_Temp"] - df["Max_Temp_Lag2"]
df["Night_Momentum_1D"] = df["Yesterday_Min_Temp"] - df["Min_Temp_Lag2"]
df["Temp_Momentum_2D"] = df["Yesterday_Max_Temp"] - df.groupby("District")["Max_Temp_C"].shift(3)
df["Night_Momentum_2D"] = df["Yesterday_Min_Temp"] - df.groupby("District")["Min_Temp_C"].shift(3)


momentum_cols = ["Temp_Momentum_1D", "Temp_Momentum_2D", "Night_Momentum_1D", "Night_Momentum_2D"]
print(df[momentum_cols].describe().round(2))

sample_cols = ["District", "Date", "Yesterday_Max_Temp", "Temp_Momentum_1D", "Yesterday_Min_Temp", "Night_Momentum_1D", "Max_Temp_C"]
df[df["District"] == "Banke"][sample_cols].iloc[5:10]


       Temp_Momentum_1D  Temp_Momentum_2D  Night_Momentum_1D  \
count         204855.00         204834.00          204855.00   
mean               0.00              0.00               0.00   
std                1.53              1.99               1.17   
min              -11.83            -15.14              -7.11   
25%               -0.73             -1.01              -0.66   
50%                0.06              0.07               0.00   
75%                0.83              1.14               0.67   
max                8.70             11.05               6.76   

       Night_Momentum_2D  
count          204834.00  
mean                0.00  
std                 1.57  
min               -11.38  
25%                -0.86  
50%                 0.00  
75%                 0.89  
max                 9.30  


,District,Date,Yesterday_Max_Temp,Temp_Momentum_1D,Yesterday_Min_Temp,Night_Momentum_1D,Max_Temp_C
5,Banke,2000-01-06,23.82,0.00,10.57,0.49,24.64
6,Banke,2000-01-07,24.64,0.82,9.83,-0.74,23.90
7,Banke,2000-01-08,23.90,-0.74,10.47,0.64,24.34
8,Banke,2000-01-09,24.34,0.44,12.85,2.38,23.12
9,Banke,2000-01-10,23.12,-1.22,12.76,-0.09,25.63


### Multi-Day Heat Spells & Cumulative Persistence (3-Day & 7-Day Rolling Windows)
- Captures persistent heat domes and sustained nocturnal thermal stress across the past 3 days and 7 days.
- Computed on lagged values grouped by District to strictly prevent forward data leakage.


In [16]:


# 1. Daytime Heat Persistence (Past 3-Day & 7-Day Afternoon Maximums)
df["Past_3D_Avg_Max_Temp"] = df.groupby("District")["Yesterday_Max_Temp"].transform(lambda s: s.rolling(3).mean()).round(2)
df["Past_7D_Avg_Max_Temp"] = df.groupby("District")["Yesterday_Max_Temp"].transform(lambda s: s.rolling(7).mean()).round(2)
df["Past_7D_Max_Temp"]     = df.groupby("District")["Yesterday_Max_Temp"].transform(lambda s: s.rolling(7).max()).round(2)

# 2. Nighttime Thermal Load (Past 3-Day & 7-Day Nocturnal Minimums + Weekly Min Floor)
df["Past_3D_Avg_Min_Temp"] = df.groupby("District")["Yesterday_Min_Temp"].transform(lambda s: s.rolling(3).mean()).round(2)
df["Past_7D_Avg_Min_Temp"] = df.groupby("District")["Yesterday_Min_Temp"].transform(lambda s: s.rolling(7).mean()).round(2)
df["Past_7D_Min_Temp"]     = df.groupby("District")["Yesterday_Min_Temp"].transform(lambda s: s.rolling(7).min()).round(2)

# Summary statistics across all 26 years
rolling_cols = [
    "Past_3D_Avg_Max_Temp", "Past_7D_Avg_Max_Temp", "Past_7D_Max_Temp",
    "Past_3D_Avg_Min_Temp", "Past_7D_Avg_Min_Temp", "Past_7D_Min_Temp"
]
print(df[rolling_cols].describe().round(2))

# Verify sample showing heat accumulation & nocturnal load in Banke
sample_cols = [
    "District", "Date", 
    "Yesterday_Max_Temp", "Past_3D_Avg_Max_Temp", "Past_7D_Avg_Max_Temp", "Past_7D_Max_Temp",
    "Yesterday_Min_Temp", "Past_3D_Avg_Min_Temp", "Past_7D_Avg_Min_Temp", "Past_7D_Min_Temp"
]
df[df["District"] == "Banke"][sample_cols].iloc[7:12]


       Past_3D_Avg_Max_Temp  Past_7D_Avg_Max_Temp  Past_7D_Max_Temp  \
count             204834.00             204750.00         204750.00   
mean                  30.89                 30.89             32.55   
std                    5.55                  5.46              5.68   
min                   13.95                 15.03             16.62   
25%                   26.81                 26.81             28.24   
50%                   30.80                 30.82             32.19   
75%                   34.67                 34.68             36.79   
max                   48.62                 48.09             48.80   

       Past_3D_Avg_Min_Temp  Past_7D_Avg_Min_Temp  Past_7D_Min_Temp  
count             204834.00             204750.00         204750.00  
mean                  19.92                 19.92             18.49  
std                    6.10                  6.06              6.46  
min                    1.09                  2.62              0.07  
25%       

,District,Date,Yesterday_Max_Temp,Past_3D_Avg_Max_Temp,Past_7D_Avg_Max_Temp,Past_7D_Max_Temp,Yesterday_Min_Temp,Past_3D_Avg_Min_Temp,Past_7D_Avg_Min_Temp,Past_7D_Min_Temp
7,Banke,2000-01-08,23.90,24.12,24.03,24.64,10.47,10.29,10.59,9.83
8,Banke,2000-01-09,24.34,24.29,24.07,24.64,12.85,11.05,10.94,9.83
9,Banke,2000-01-10,23.12,23.79,23.88,24.64,12.76,12.03,11.16,9.83
10,Banke,2000-01-11,25.63,24.36,24.18,25.63,13.72,13.11,11.47,9.83
11,Banke,2000-01-12,26.92,25.22,24.62,26.92,14.53,13.67,12.10,9.83


### Soil Moisture & Drought Deficit Dynamics (Cumulative Rain & Consecutive Dry Days)
- Models the land-atmosphere drought feedback loop: dry soil eliminates evaporative cooling, triggering runaway sensible heating.
- Formulated using WMO ETCCDI standards (Consecutive Dry Days < 1.0 mm) on historical lagged precipitation.


In [17]:
# Step 12: Soil Moisture & Drought Deficit - Cumulative Rain & Consecutive Dry Days (CDD)

# 1. Multi-Scale Cumulative Rainfall Deficits (Past 7-Day & 14-Day Rain Sums)
df["Past_7D_Rain_Sum"]  = df.groupby("District")["Yesterday_Precip"].transform(lambda s: s.rolling(7).sum()).round(2)
df["Past_14D_Rain_Sum"] = df.groupby("District")["Yesterday_Precip"].transform(lambda s: s.rolling(14).sum()).round(2)

# 2. Consecutive Dry Days (CDD) - WMO ETCCDI International Standard (< 1.0 mm)
# A running counter of dry days that resets to 0 as soon as a rain event (>= 1.0 mm) occurs
df["Consecutive_Dry_Days"] = df.groupby("District")["Yesterday_Precip"].transform(
    lambda s: (s < 1.0).groupby((s >= 1.0).cumsum()).cumsum()
)

# Summary statistics across 26 years
drought_cols = ["Past_7D_Rain_Sum", "Past_14D_Rain_Sum", "Consecutive_Dry_Days"]
print("Drought Deficit & Soil Moisture Statistics:")
print(df[drought_cols].describe().round(2))

# Verify sample showing dry spell accumulation & rain reset in Banke
sample_cols = ["District", "Date", "Yesterday_Precip", "Past_7D_Rain_Sum", "Past_14D_Rain_Sum", "Consecutive_Dry_Days"]
df[df["District"] == "Banke"][sample_cols].iloc[10:16]


Drought Deficit & Soil Moisture Statistics:
       Past_7D_Rain_Sum  Past_14D_Rain_Sum  Consecutive_Dry_Days
count         204750.00          204603.00             204897.00
mean              25.74              51.49                 15.16
std               42.06              73.25                 25.47
min                0.00               0.00                  0.00
25%                0.03               0.66                  0.00
50%                5.36              16.53                  3.00
75%               34.97              77.56                 19.00
max              449.10             635.70                188.00


,District,Date,Yesterday_Precip,Past_7D_Rain_Sum,Past_14D_Rain_Sum,Consecutive_Dry_Days
10,Banke,2000-01-11,0.05,0.07,NaN,10
11,Banke,2000-01-12,0.00,0.07,NaN,11
12,Banke,2000-01-13,1.03,1.10,NaN,0
13,Banke,2000-01-14,8.83,9.93,NaN,0
14,Banke,2000-01-15,0.00,9.93,9.93,1
15,Banke,2000-01-16,0.00,9.93,9.93,2


### Duration Dynamics — Heatwave streak
- Tracks consecutive days of extreme heat (>= 40.0°C).


In [18]:

df["Heatwave_Streak"] = df.groupby("District")["Yesterday_Max_Temp"].transform(
    lambda s: (s >= 40.0).groupby((s < 40.0).cumsum()).cumsum()
)

print(df["Heatwave_Streak"].describe().round(2))
max_streak_idx = df["Heatwave_Streak"].idxmax()
longest_dist = df.loc[max_streak_idx, "District"]
longest_date = df.loc[max_streak_idx, "Date"].strftime('%Y-%m-%d')
print(f"\nLongest Historical Heatwave: {df.loc[max_streak_idx, 'Heatwave_Streak']} consecutive days in {longest_dist} (ending {longest_date})")

sample_cols = ["District", "Date", "Yesterday_Max_Temp", "Heatwave_Streak", "Max_Temp_C"]
df[df["District"] == "Banke"][sample_cols].iloc[100:106]


count    204897.00
mean          0.39
std           2.20
min           0.00
25%           0.00
50%           0.00
75%           0.00
max          43.00
Name: Heatwave_Streak, dtype: float64

Longest Historical Heatwave: 43 consecutive days in Bardiya (ending 2012-06-19)


,District,Date,Yesterday_Max_Temp,Heatwave_Streak,Max_Temp_C
100,Banke,2000-04-10,37.33,0,38.20
101,Banke,2000-04-11,38.20,0,39.09
102,Banke,2000-04-12,39.09,0,40.27
103,Banke,2000-04-13,40.27,1,40.02
104,Banke,2000-04-14,40.02,2,40.11
105,Banke,2000-04-15,40.11,3,39.10


### 26-Year Climatological Baseline & Statistical Anomalies (Departures & Z-Scores)
- Establishes a localized 2000–2026 baseline normal (mean & std) for each district on every calendar day of the year.
- Computes temperature departures (Delta T_clim) and Gaussian Z-scores across both daytime highs and nocturnal lows.


In [19]:
# Step 14: Climatological Baseline & Statistical Anomalies (Day & Night)

# Ensure Day_of_Year is present
if "Day_of_Year" not in df.columns:
    df["Day_of_Year"] = df["Date"].dt.dayofyear

# 1. Multi-Decadal Historical Baseline Normal (Mean & Std per District on exact calendar Day of Year)
df["Clim_Max_Mean"] = df.groupby(["District", "Day_of_Year"])["Max_Temp_C"].transform("mean").round(2)  
df["Clim_Max_Std"]  = df.groupby(["District", "Day_of_Year"])["Max_Temp_C"].transform("std").round(2)
df["Clim_Min_Mean"] = df.groupby(["District", "Day_of_Year"])["Min_Temp_C"].transform("mean").round(2)
df["Clim_Min_Std"]  = df.groupby(["District", "Day_of_Year"])["Min_Temp_C"].transform("std").round(2)

# 2. Climatological Temperature Anomalies (Degree departure from 26-year seasonal normal)
df["Temp_Anomaly"]  = (df["Yesterday_Max_Temp"] - df["Clim_Max_Mean"]).round(2)
df["Night_Anomaly"] = (df["Yesterday_Min_Temp"] - df["Clim_Min_Mean"]).round(2)

# 3. Standardized Statistical Z-Scores (Standard deviations from normal)
df["Temp_Z_Score"]  = ((df["Yesterday_Max_Temp"] - df["Clim_Max_Mean"]) / (df["Clim_Max_Std"] + 1e-5)).round(2)
df["Night_Z_Score"] = ((df["Yesterday_Min_Temp"] - df["Clim_Min_Mean"]) / (df["Clim_Min_Std"] + 1e-5)).round(2)

# Summary statistics across all 26 years
anomaly_cols = ["Clim_Max_Mean", "Temp_Anomaly", "Temp_Z_Score", "Clim_Min_Mean", "Night_Anomaly", "Night_Z_Score"]
print("Climatological Baseline & Anomaly Statistics:")
print(df[anomaly_cols].describe().round(2))

# Verify sample showing early-season heat anomalies in Banke (March/April)
sample_cols = ["District", "Date", "Yesterday_Max_Temp", "Clim_Max_Mean", "Temp_Anomaly", "Temp_Z_Score", "Yesterday_Min_Temp", "Night_Anomaly", "Night_Z_Score"]
df[df["District"] == "Banke"][sample_cols].iloc[70:75]


Climatological Baseline & Anomaly Statistics:
       Clim_Max_Mean  Temp_Anomaly  Temp_Z_Score  Clim_Min_Mean  \
count      204897.00     204876.00     204876.00      204897.00   
mean           30.88         -0.00         -0.00          19.92   
std             5.14          2.32          1.01           5.94   
min            19.33        -16.16         -5.61           7.70   
25%            26.86         -1.33         -0.63          14.38   
50%            30.95          0.03          0.02          21.71   
75%            34.76          1.42          0.65          25.23   
max            42.09         12.16          5.85          29.32   

       Night_Anomaly  Night_Z_Score  
count      204876.00      204876.00  
mean           -0.00           0.01  
std             1.63           1.01  
min            -9.16          -4.76  
25%            -0.94          -0.67  
50%             0.01           0.01  
75%             0.96           0.68  
max             7.81           5.67  


,District,Date,Yesterday_Max_Temp,Clim_Max_Mean,Temp_Anomaly,Temp_Z_Score,Yesterday_Min_Temp,Night_Anomaly,Night_Z_Score
70,Banke,2000-03-11,30.42,31.13,-0.71,-0.26,13.53,-2.88,-1.36
71,Banke,2000-03-12,32.23,31.87,0.36,0.14,14.96,-2.00,-0.92
72,Banke,2000-03-13,32.69,31.08,1.61,0.51,15.21,-2.28,-1.12
73,Banke,2000-03-14,32.34,31.15,1.19,0.39,17.94,0.69,0.33
74,Banke,2000-03-15,31.15,31.69,-0.54,-0.19,15.19,-1.56,-0.53


### Earth's Orbital Geometry & Cyclical Time Features (Trigonometric sin/cos & Seasons)
- Projects annual Day-of-Year and Monthly cycles onto a 2D continuous unit circle using Sine and Cosine transformations.
- Eliminates the artificial Dec 31 / Jan 1 boundary discontinuity (Euclidean distance = 0.0215).
- Encodes Nepal's 4 official Department of Hydrology & Meteorology (DHM) meteorological seasons.


In [20]:
# Step 15: Cyclical Time & Seasonal Encodings (Earth's Orbital Geometry)

df["Month"] = df["Date"].dt.month
if "Day_of_Year" not in df.columns:
    df["Day_of_Year"] = df["Date"].dt.dayofyear

# 1. Continuous Trigonometric Sine and Cosine Encodings
# Day of Year Cycle (Period = 365.25 days to accommodate astronomical leap-year orbit)
df["DOY_Sin"] = np.sin(2 * np.pi * df["Day_of_Year"] / 365.25).round(4)
df["DOY_Cos"] = np.cos(2 * np.pi * df["Day_of_Year"] / 365.25).round(4)

# Monthly Cycle (Period = 12 months)
df["Month_Sin"] = np.sin(2 * np.pi * df["Month"] / 12.0).round(4)
df["Month_Cos"] = np.cos(2 * np.pi * df["Month"] / 12.0).round(4)

# 2. Nepal Department of Hydrology & Meteorology (DHM) 4 Official Seasons
season_conditions = [
    df["Month"].isin([12, 1, 2]),       # Winter
    df["Month"].isin([3, 4, 5]),        # Pre-Monsoon (Primary Heatwave Window)
    df["Month"].isin([6, 7, 8, 9]),     # Monsoon
    df["Month"].isin([10, 11])          # Post-Monsoon
]
season_choices = ["Winter", "Pre-Monsoon", "Monsoon", "Post-Monsoon"]
df["Season"] = np.select(season_conditions, season_choices, default="Winter")

# Summary check: Verify season frequencies and boundary continuity
print("Nepal DHM Seasonal Distribution:")
print(df["Season"].value_counts())

# Sample verification across different seasons
sample_cols = ["District", "Date", "Day_of_Year", "DOY_Sin", "DOY_Cos", "Month", "Month_Sin", "Month_Cos", "Season"]
df[sample_cols].iloc[[0, 100, 200, 300]]


Nepal DHM Seasonal Distribution:
Season
Monsoon         68901
Pre-Monsoon     52164
Winter          50526
Post-Monsoon    33306
Name: count, dtype: int64


,District,Date,Day_of_Year,DOY_Sin,DOY_Cos,Month,Month_Sin,Month_Cos,Season
0,Banke,2000-01-01,1,0.0172,0.9999,1,0.500,0.866,Winter
100,Banke,2000-04-10,101,0.9861,-0.1659,4,0.866,-0.500,Pre-Monsoon
200,Banke,2000-07-19,201,-0.3109,-0.9505,7,-0.500,-0.866,Monsoon
300,Banke,2000-10-27,301,-0.8936,0.4489,10,-0.866,0.500,Post-Monsoon


### Statistical Distribution Corrections — Logarithmic Rainfall Compression (log1p)
- Applies non-linear monotonic natural logarithm compression ln(1 + x) to zero-inflated, highly right-skewed precipitation.
- Reduces daily rainfall skewness from 5.70 to 1.26, and 7-day rain skewness from 2.63 to 0.23.
- Prevents gradient explosion in neural networks and linear models while preserving extreme storm events.


In [21]:
# Step 16: Statistical Distribution Corrections - Log-Transformed Rainfall (np.log1p)

# 1. Log-transformed Daily Rainfall (Today & Yesterday)
df["Rain_Log"] = np.log1p(df["Precipitation_mm"]).round(4)
df["Yesterday_Rain_Log"] = np.log1p(df["Yesterday_Precip"]).round(4)

# 2. Log-transformed 7-Day Cumulative Rainfall
df["Past_7D_Rain_Log"] = np.log1p(df["Past_7D_Rain_Sum"]).round(4)

# Verify skewness reduction
raw_skew = df["Precipitation_mm"].skew()
log_skew = df["Rain_Log"].skew()
raw_7d_skew = df["Past_7D_Rain_Sum"].skew()
log_7d_skew = df["Past_7D_Rain_Log"].skew()

print(f"Daily Precipitation Skewness: Raw = {raw_skew:.2f} --> Log1p = {log_skew:.2f}")
print(f"Past 7D Rain Sum Skewness:   Raw = {raw_7d_skew:.2f} --> Log1p = {log_7d_skew:.2f}")

# Summary statistics across 26 years
rain_cols = ["Precipitation_mm", "Rain_Log", "Yesterday_Precip", "Yesterday_Rain_Log", "Past_7D_Rain_Sum", "Past_7D_Rain_Log"]
print("\nLog-Transformed Rainfall Statistics:")
print(df[rain_cols].describe().round(2))

# Verify sample showing dry vs heavy rain days in Banke
sample_cols = ["District", "Date", "Precipitation_mm", "Rain_Log", "Yesterday_Precip", "Yesterday_Rain_Log", "Past_7D_Rain_Sum", "Past_7D_Rain_Log"]
df[df["District"] == "Banke"][sample_cols].iloc[12:17]


Daily Precipitation Skewness: Raw = 5.70 --> Log1p = 1.26
Past 7D Rain Sum Skewness:   Raw = 2.63 --> Log1p = 0.23

Log-Transformed Rainfall Statistics:
       Precipitation_mm   Rain_Log  Yesterday_Precip  Yesterday_Rain_Log  \
count         204897.00  204897.00         204876.00           204876.00   
mean               3.68       0.77              3.68                0.77   
std                9.03       1.06              9.03                1.06   
min                0.00       0.00              0.00                0.00   
25%                0.00       0.00              0.00                0.00   
50%                0.08       0.08              0.08                0.08   
75%                3.14       1.42              3.14                1.42   
max              272.13       5.61            272.13                5.61   

       Past_7D_Rain_Sum  Past_7D_Rain_Log  
count         204750.00         204750.00  
mean              25.74              1.97  
std               42.06       

,District,Date,Precipitation_mm,Rain_Log,Yesterday_Precip,Yesterday_Rain_Log,Past_7D_Rain_Sum,Past_7D_Rain_Log
12,Banke,2000-01-13,8.83,2.2854,1.03,0.7080,1.10,0.7419
13,Banke,2000-01-14,0.00,0.0000,8.83,2.2854,9.93,2.3915
14,Banke,2000-01-15,0.00,0.0000,0.00,0.0000,9.93,2.3915
15,Banke,2000-01-16,0.00,0.0000,0.00,0.0000,9.93,2.3915
16,Banke,2000-01-17,0.00,0.0000,0.00,0.0000,9.91,2.3897


### Spatial Geography & Macro-Climatic Regionalization
- Captures the 800-km West-to-East continental climate divide using continuous coordinates (Latitude, Longitude).
- Formulates a binary high-hazard western belt indicator (Is_Western_Terai) and 3-tier macro-zones (Western, Central, Eastern).
- Reflects continental desert heat advection (8.63% heatwave rate) vs. eastern maritime moisture moderation (5.27% rate).


In [22]:
# Step 17: Spatial Geography & Macro-Climatic Regionalization

# 1. Binary Western Terai Hazard Belt Indicator (Lumbini & Sudurpashchim)
df["Is_Western_Terai"] = df["Province"].isin(["Lumbini", "Sudurpashchim"]).astype(int)

# 2. Three-Tier Macro-Climatic Regional Classification
region_conditions = [
    df["Province"].isin(["Lumbini", "Sudurpashchim"]),
    df["Province"].isin(["Gandaki", "Bagmati"]),
    df["Province"].isin(["Madhesh", "Koshi"])
]
region_choices = ["Western Terai", "Central Terai", "Eastern Terai"]
df["Terai_Region"] = np.select(region_conditions, region_choices, default="Eastern Terai")

# Verify regional distribution and geographical divide
print("Western Terai Belt Distribution (%):")
print((df["Is_Western_Terai"].value_counts(normalize=True) * 100).round(2))

print("\nMacro-Region Observation Counts:")
print(df["Terai_Region"].value_counts())

# Empirical proof of spatial heat gradient
print("\nEmpirical Heat Gradient by Macro-Region:")
spatial_summary = df.groupby("Terai_Region").agg(
    Districts=("District", "nunique"),
    Mean_Max_Temp=("Max_Temp_C", "mean"),
    Max_Recorded_Temp=("Max_Temp_C", "max"),
    Heatwave_Pct=("Heatwave_Flag", lambda x: (x.mean() * 100).round(2)),
    Mean_Longitude=("Longitude", "mean")
).sort_values("Mean_Longitude")
print(spatial_summary.round(2))

# Sample rows
sample_cols = ["District", "Province", "Latitude", "Longitude", "Is_Western_Terai", "Terai_Region", "Max_Temp_C"]
df[sample_cols].iloc[[0, 50000, 100000, 150000]]


Western Terai Belt Distribution (%):
Is_Western_Terai
0    61.9
1    38.1
Name: proportion, dtype: float64

Macro-Region Observation Counts:
Terai_Region
Eastern Terai    107327
Western Terai     78056
Central Terai     19514
Name: count, dtype: int64

Empirical Heat Gradient by Macro-Region:
               Districts  Mean_Max_Temp  Max_Recorded_Temp  Heatwave_Pct  \
Terai_Region                                                               
Western Terai          8          30.79              48.80          8.63   
Central Terai          2          30.37              44.87          4.17   
Eastern Terai         11          31.04              46.41          5.27   

               Mean_Longitude  
Terai_Region                   
Western Terai           82.05  
Central Terai           84.28  
Eastern Terai           86.17  


,District,Province,Latitude,Longitude,Is_Western_Terai,Terai_Region,Max_Temp_C
0,Banke,Lumbini,28.05,81.62,1,Western Terai,24.02
50000,Dhanusha,Madhesh,26.73,85.92,0,Eastern Terai,35.63
100000,Mahottari,Madhesh,26.64,85.80,0,Eastern Terai,32.41
150000,Rautahat,Madhesh,26.76,85.28,0,Eastern Terai,22.34


### Step 18: Quality Assurance, Warmup NaN Trimming & Final Dataset Export

#### 1. Mathematical Geometry of Boundary Missing Values
In causal time-series engineering, backward rolling windows and forward-shifted prediction targets naturally create boundary missing values:
1. **Leading Warmup Boundary (14 Days per District):** Our longest cumulative drought rolling window spans 14 days (`Past_14D_Rain_Sum`). Consequently, the first 14 calendar dates of the archive (January 1 to January 14, 2000) lack sufficient prior history. Across all 21 districts, this accounts for exactly $14 \times 21 = 294$ rows.
2. **Trailing Lookahead Boundary (1 Day per District):** The continuous prognostic target `Next_Day_Max_Temp` requires a 1-day forward shift ($t+1$). The final recorded calendar date (September 17, 2026) has no observable subsequent date. Across all 21 districts, this accounts for exactly $1 \times 21 = 21$ rows.

#### 2. Trimming vs. Imputation Rationale
Imputing early-archive warmup values (e.g., using mean temperatures or zero precipitation) would inject artificial noise and distort early-season model training. Because the raw dataset comprises 204,897 records, trimming these boundary rows removes only $294 + 21 = 315$ rows in total—retaining **204,582 clean records (99.85% retention)** with **0 missing values**.

#### 3. Prediction Target Scope
To maintain a focused modeling objective, the feature matrix retains strictly **3 Heatwave Prediction Targets**:
- **Target 1 (Binary Classification):** `Heatwave_Flag` ($\ge 40.0^\circ\text{C}$)
- **Target 2 (Multi-Class Severity):** `Heatwave_Severity` (Tiers 0 to 3)
- **Target 3 (Continuous Regression):** `Next_Day_Max_Temp` (Tomorrow's afternoon high)

All nocturnal thermal metrics, wind indices, and atmospheric anomalies are preserved and engineered as rich input features.


In [ ]:

import os

# dropping next dat min temp because our main focus is future  max temp not min
if "Next_Day_Min_Temp" in df.columns:
    df = df.drop(columns=["Next_Day_Min_Temp"])
    
# drop null 
null_summary = df.isnull().sum()
print("Shape of df after featureEngeneering",df.shape)
# print(null_summary)
columns_with_nulls = null_summary[null_summary > 0]
df_final = df.dropna().reset_index(drop=True)
print("Shape of df adter drop some nan",df_final.shape)


# import sys
# sys.exit()

# 5. Target Variable Distribution Validation
print("\n--- TARGET VARIABLE DISTRIBUTIONS ---")
print("Target 1: Binary Heatwave Alert (Heatwave_Flag):")
print(df_final["Heatwave_Flag"].value_counts())
print((df_final["Heatwave_Flag"].value_counts(normalize=True) * 100).round(2).rename("Percentage (%)"))

print("\nTarget 2: Tiered Heatwave Severity (Heatwave_Severity):")
print(df_final["Heatwave_Severity"].value_counts())
print((df_final["Heatwave_Severity"].value_counts(normalize=True) * 100).round(2).rename("Percentage (%)"))

print("\nTarget 3: Tomorrow's High Temperature (Next_Day_Max_Temp, °C):")
print(df_final["Next_Day_Max_Temp"].describe().round(2))

# 6. Export Final Feature Engineered Dataset
# output_dir = "../data/processed"
# os.makedirs(output_dir, exist_ok=True)
# export_path = os.path.join(output_dir, "terai_features_engineered.csv")

# print(f"\nExporting finalized feature matrix to: {export_path} ...")
# df_final.to_csv(export_path, index=False)

# file_size_mb = os.path.getsize(export_path) / (1024 * 1024)
# print(f"✅ Successfully exported 'terai_features_engineered.csv' ({file_size_mb:.2f} MB on disk)")


Shape of df after featureEngeneering (204897, 65)
Shape of df adter drop some nan (204582, 65)

--- TARGET VARIABLE DISTRIBUTIONS ---
Target 1: Binary Heatwave Alert (Heatwave_Flag):
Heatwave_Flag
0    191378
1     13204
Name: count, dtype: int64
Heatwave_Flag
0    93.55
1     6.45
Name: Percentage (%), dtype: float64

Target 2: Tiered Heatwave Severity (Heatwave_Severity):
Heatwave_Severity
0    191378
1      8205
2      4600
3       399
Name: count, dtype: int64
Heatwave_Severity
0    93.55
1     4.01
2     2.25
3     0.20
Name: Percentage (%), dtype: float64

Target 3: Tomorrow's High Temperature (Next_Day_Max_Temp, °C):
count    204582.00
mean         30.89
std           5.63
min          11.77
25%          26.80
50%          30.78
75%          34.72
max          48.80
Name: Next_Day_Max_Temp, dtype: float64

Exporting finalized feature matrix to: ../data/processed\terai_features_engineered.csv ...
✅ Successfully exported 'terai_features_engineered.csv' (88.29 MB on disk)
